《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구입할 수 있습니다. 두 링크는 제휴 링크입니다. 이 책이 도움이 되었다면 Green Tea Press가 제공하는 [무료 공개 교육 자료를 후원](https://greenteapress.com/support/?book=ts3)하는 것도 고려해 주세요.

# 탐색적 자료 분석

이 책의 중심 생각은 데이터를 이용해 질문에 답하고, 논쟁을 해결하고, 더 나은 결정을 내릴 수 있다는 것입니다.

이 장에서는 이를 위한 단계를 소개합니다. 데이터를 불러와 검증하고, 탐색한 다음, 관심 있는 것을 측정할 통계량을 고르는 일입니다. 예제로는 미국 가족성장조사(National Survey of Family Growth, NSFG) 자료를 사용합니다. 아내와 제가 첫아이를 기다릴 때 들었던 질문, “첫아이는 늦게 태어나는 경향이 있을까?”에 답해 보겠습니다.

[이 노트북의 원문을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/v3/nb/chap01.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import HTML
from thinkstats import decorate

## 근거

첫아이는 늦게 태어날 가능성이 더 높다는 말을 들어 봤을 것입니다. 이 질문으로 인터넷을 검색하면 많은 논의를 찾을 수 있습니다. 맞는 말이라는 사람도 있고, 근거 없는 이야기라는 사람도 있습니다. 반대로 첫아이가 일찍 태어난다는 사람도 있습니다.

이런 논의에서 사람들은 자기 주장을 뒷받침하려고 데이터를 제시하곤 합니다. 저는 다음과 같은 예를 많이 보았습니다.

> “최근에 첫아이를 낳은 제 친구 두 명은 둘 다 예정일을 거의 2주나 넘긴 뒤에야 진통이 시작되거나 유도분만을 했어요.”

> “첫아이는 2주 늦게 나왔는데, 지금은 둘째가 2주 일찍 나올 것 같아요!!”

> “그건 사실이 아닌 것 같아요. 언니는 엄마의 첫아이였는데 일찍 태어났거든요. 제 사촌들도 그런 경우가 많고요.”

이런 이야기를 **일화적 근거**(anecdotal evidence)라고 합니다. 공개되지 않은, 대개 개인적인 데이터에 바탕을 두기 때문입니다. 일상 대화에서 경험담을 나누는 것은 아무 문제가 없습니다. 인용한 사람들을 탓하려는 뜻은 아닙니다.

하지만 우리는 좀 더 설득력 있는 근거와 믿을 만한 답을 원할 수 있습니다. 그런 기준에서 일화적 근거는 대개 부족합니다. 이유는 다음과 같습니다.

- 관측 수가 적음: 첫아이의 임신 기간이 더 길더라도, 그 차이는 자연적으로 생기는 변동에 비하면 작을 가능성이 높습니다. 그렇다면 차이가 있는지 알려면 많은 임신 사례를 비교해야 할 수 있습니다.
- 선택 편향: 이 논의에 참여하는 사람들은 자기 첫아이가 늦게 태어났기 때문에 관심을 가졌을 수 있습니다. 그렇다면 데이터를 고르는 과정에서 결과가 한쪽으로 치우치게 됩니다.
- 확증 편향: 주장을 믿는 사람은 이를 뒷받침하는 예를 더 잘 제시할 수 있습니다. 의심하는 사람은 반례를 더 잘 들 것입니다.
- 부정확성: 경험담은 흔히 개인적인 이야기입니다. 잘못 기억하거나, 사실과 다르게 말하거나, 전달하는 과정에서 부정확해질 수 있습니다.

경험담의 한계를 보완하기 위해 다음과 같은 통계학의 도구를 사용하겠습니다.

- 자료 수집: 미국 인구에 대해 통계적으로 타당한 추론을 할 목적으로 설계된 대규모 전국 조사 자료를 사용합니다.
- 기술통계: 데이터를 간결하게 요약하는 통계량을 구하고, 데이터를 시각화하는 여러 방법을 살펴봅니다.
- 탐색적 자료 분석: 관심 있는 질문과 관련된 패턴, 차이, 그 밖의 특징을 찾습니다. 동시에 앞뒤가 맞지 않는 부분을 확인하고 한계를 파악합니다.
- 추정: 표본 자료로 모집단의 특성을 추정합니다.
- 가설검정: 두 집단의 차이처럼 눈에 띄는 효과가 보이면, 그 효과가 우연히 생겼을 가능성을 평가합니다.

함정을 피하도록 주의하면서 이 단계를 수행하면, 근거가 더 탄탄하고 옳을 가능성도 더 높은 결론에 이를 수 있습니다.

## 미국 가족성장조사

미국 질병통제예방센터(CDC)는 1973년부터 미국 가족성장조사(NSFG)를 실시해 왔습니다. 조사 목적은 “가족생활, 결혼과 이혼, 임신, 난임, 피임법 사용, 남녀의 건강에 관한 정보”를 모으는 것입니다. 조사 결과는 “보건 서비스와 보건 교육 프로그램을 계획하고, 가족·출산·건강에 관한 통계 연구를 수행하는 데” 쓰입니다.

NSFG에 관한 자세한 내용은 [CDC의 NSFG 안내](https://www.cdc.gov/nchs/nsfg/)에서 볼 수 있습니다.

이 조사 자료로 첫아이가 늦게 태어나는 경향이 있는지와 그 밖의 질문들을 살펴보겠습니다. 자료를 제대로 사용하려면 조사 설계를 이해해야 합니다.

일반적으로 통계 연구의 목적은 **모집단**(population)에 관한 결론을 내리는 것입니다. NSFG의 목표 모집단은 미국에 사는 15~44세 사람들입니다.

이상적으로는 모집단의 모든 구성원에게서 자료를 수집하면 좋겠지만, 대개는 불가능합니다. 대신 모집단의 일부인 **표본**(sample)에서 자료를 수집합니다. 조사에 참여하는 사람들을 **응답자**(respondents)라고 합니다.

NSFG는 **횡단면 연구**(cross-sectional study)입니다. 특정 시점의 모집단 모습을 포착한다는 뜻입니다. NSFG는 여러 차례 실시되었으며, 한 차례의 조사를 **주기**(cycle)라고 부릅니다. 여기서는 2002년 1월부터 2003년 3월까지 실시된 제6주기 자료를 사용하겠습니다.

횡단면 연구는 보통 **대표성**(representativeness)을 갖추는 것을 목표로 합니다. 연구 목적에 중요한 모든 면에서 표본이 목표 모집단과 비슷해야 한다는 뜻입니다. 실제로 이 이상을 달성하기는 어렵지만, 조사 담당자들은 최대한 가까워지도록 노력합니다.

NSFG는 일부 집단을 의도적으로 **과대표집**(oversampling)합니다. 그 집단에 속한 사람이 표본에 뽑힐 확률을 더 높인다는 뜻입니다. 조사 설계자들은 히스패닉, 아프리카계 미국인, 십 대의 세 집단을 미국 인구에서 차지하는 비율보다 더 많이 모집했습니다. 각 집단에서 타당한 결론을 내릴 수 있을 만큼 충분한 응답자를 확보하기 위해서입니다. 과대표집의 단점은 표본 통계량으로 모집단에 관한 결론을 내리기가 그만큼 까다로워진다는 점입니다. 이 문제는 나중에 다시 다루겠습니다.

이런 자료를 다룰 때는 **코드북**(codebook)을 잘 알아 두어야 합니다. 코드북에는 조사 설계, 설문 문항, 응답을 부호화한 방식이 기록되어 있습니다.

NSFG 자료의 코드북과 사용자 안내서는 [CDC의 제6주기 자료 안내](https://www.cdc.gov/nchs/nsfg/nsfg_cycle6.htm)에서 구할 수 있습니다.

## 데이터 읽기

NSFG 자료를 내려받기 전에 다음 이용 조건에 동의해야 합니다.

> 개인이나 기관을 의도적으로 식별하거나 공개하는 행위는 정보 제공자에게 약속한 비밀 보장을 위반합니다. 따라서 이용자는 다음을 지켜야 합니다.
>
> - 이 데이터셋의 자료를 통계 보고와 분석 목적으로만 사용합니다.
> - 자료에 포함된 개인이나 기관의 신원을 알아내려 하지 않습니다.
> - 이 데이터셋을 NCHS 또는 그 밖의 데이터셋에 있는 개인 식별 가능 자료와 연결하지 않습니다.
> - 개인과 기관을 보호하기 위해 적용한 공개 제한 방법을 평가하려 하거나, 개인과 기관을 재식별하는 방법을 연구하지 않습니다.

이 조건을 지키는 데 동의한다면, 이 장의 노트북에 있는 안내에 따라 자료를 내려받을 수 있습니다.

데이터 파일은 [NSFG 웹사이트](https://www.cdc.gov/nchs/data_access/ftp_dua.htm?url_redirect=ftp://ftp.cdc.gov/pub/Health_Statistics/NCHS/Datasets/NSFG)에서 직접 구할 수 있습니다. 여기서는 데이터 파일의 압축본을 제공하는 이 책의 저장소에서 내려받겠습니다.

다음 셀들은 데이터 파일을 내려받고, 파일을 읽는 데 필요한 `statadict`를 설치합니다.

In [4]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

In [5]:
try:
    import statadict
except ImportError:
    %pip install statadict

자료는 두 파일에 저장되어 있습니다. 데이터 형식을 설명하는 “사전(dictionary)” 파일과 데이터 파일입니다.

In [6]:
dct_file = "2002FemPreg.dct"
dat_file = "2002FemPreg.dat.gz"

이 장의 노트북에는 두 파일을 읽는 함수가 정의되어 있습니다. 데이터 형식이 Stata라는 통계 소프트웨어와 호환되기 때문에 함수 이름을 `read_stata`로 지었습니다.

다음 함수는 두 파일 이름을 인수로 받습니다. 사전 파일을 읽고, 그 결과를 이용해 데이터 파일을 읽습니다.

In [7]:
from statadict import parse_stata_dict


def read_stata(dct_file, dat_file):
    stata_dict = parse_stata_dict(dct_file)
    resp = pd.read_fwf(
        dat_file,
        names=stata_dict.names,
        colspecs=stata_dict.colspecs,
        compression="gzip",
    )
    return resp

다음과 같이 사용합니다.

In [8]:
preg = read_stata(dct_file, dat_file)

결과는 `DataFrame`입니다. 행과 열로 된 표 형식 자료를 나타내는 pandas 자료 구조입니다. 이 `DataFrame`에서는 응답자가 보고한 임신 한 건이 행 하나에 해당하고, **변수**(variable) 하나가 열 하나에 해당합니다. 변수에는 설문 문항에 대한 응답이나, 하나 이상의 응답으로 계산한 값이 들어갈 수 있습니다.

`DataFrame`에는 데이터 외에 변수 이름과 자료형도 들어 있습니다. 데이터에 접근하고 수정하는 메서드도 제공합니다. `shape` 속성에는 행과 열의 수가 들어 있습니다.

In [9]:
preg.shape

이 데이터셋에는 임신 13,593건에 관한 정보가 243개 변수에 담겨 있습니다. `DataFrame`의 `head` 메서드는 처음 몇 행을 보여 줍니다.

In [10]:
preg.head()

맨 왼쪽 열은 `DataFrame`의 인덱스로, 각 행의 이름표를 담고 있습니다. 여기서는 0부터 시작하는 정수이지만, 문자열이나 다른 자료형도 사용할 수 있습니다.

`DataFrame`의 `columns` 속성에는 변수 이름이 들어 있습니다.

In [11]:
preg.columns

열 이름은 pandas의 또 다른 자료 구조인 `Index` 객체에 담겨 있습니다. `DataFrame`에서 열에 접근하려면 열 이름을 키로 사용하면 됩니다.

In [12]:
pregordr = preg["pregordr"]
type(pregordr)

결과는 값의 나열을 나타내는 pandas `Series`입니다. `Series`도 처음 몇 개의 값과 그 이름표를 보여 주는 `head`를 제공합니다.

In [13]:
pregordr.head()

마지막 줄에는 `Series`의 이름과 값의 자료형인 `dtype`이 나옵니다. 여기서 `int64`는 값이 64비트 정수라는 뜻입니다.

NSFG 데이터셋에는 모두 243개 변수가 있습니다. 이 책의 탐색에서 사용할 변수 중 일부는 다음과 같습니다.

- `caseid`: 응답자의 정수형 식별 번호입니다.
- `pregordr`: 임신 순서입니다. 한 응답자의 첫 번째 임신은 1, 두 번째는 2처럼 부호화되어 있습니다.
- `prglngth`: 임신 기간을 주 단위 정수로 나타냅니다.
- `outcome`: 임신 결과를 나타내는 정수 코드입니다. 1은 생아 출산을 뜻합니다.
- `birthord`: 생아 출산 순서입니다. 한 응답자의 첫아이는 1이고, 이후에도 같은 식입니다. 생아 출산 이외의 결과에서는 이 칸이 비어 있습니다.
- `birthwgt_lb`, `birthwgt_oz`: 아기의 출생 체중 중 파운드 부분과 온스 부분입니다.
- `agepreg`: 임신이 끝났을 때의 산모 나이입니다.
- `finalwgt`: 응답자에게 부여된 표본 가중치입니다. 이 응답자가 미국 모집단의 몇 명을 대표하는지 나타내는 부동소수점 값입니다.

코드북을 자세히 읽어 보면, 많은 변수가 **재코딩 변수**(recode)라는 것을 알 수 있습니다. 조사에서 수집한 **원자료**(raw data) 자체가 아니라, 원자료로 계산한 변수라는 뜻입니다.

예를 들어 생아 출산의 `prglngth`는 원자료 변수 `wksgest`(임신 주수)가 있으면 그 값을 사용합니다. 없으면 `mosgest * 4.33`, 즉 임신 개월 수에 한 달의 평균 주수를 곱해 추정합니다.

재코딩에는 데이터의 일관성과 정확성을 확인하는 로직이 쓰이는 경우가 많습니다. 원자료를 직접 처리해야 할 분명한 이유가 없다면, 재코딩 변수가 있을 때는 그것을 사용하는 편이 대체로 좋습니다.

## 검증

한 소프트웨어 환경에서 데이터를 내보내 다른 환경으로 가져오는 과정에서 오류가 생길 수 있습니다. 새로운 데이터셋을 익힐 때는 값을 잘못 해독하거나 의미를 오해할 수도 있습니다. 검증에 시간을 들이면 나중에 시간을 절약하고 오류도 피할 수 있습니다.

검증하는 한 가지 방법은 기본 통계량을 계산해 공개된 결과와 비교하는 것입니다. 예를 들어 NSFG 코드북에는 각 변수를 요약한 표가 있습니다. 다음은 임신 결과를 부호화한 `outcome`의 표입니다.

In [14]:
def show_table(d):
    df = pd.DataFrame(d)
    return HTML(df.to_html(index=False))

In [15]:
d = {
    "Value": [1, 2, 3, 4, 5, 6, "Total"],
    "Label": [
        "LIVE BIRTH",
        "INDUCED ABORTION",
        "STILLBIRTH",
        "MISCARRIAGE",
        "ECTOPIC PREGNANCY",
        "CURRENT PREGNANCY",
        "",
    ],
    "Total": [9148, 1862, 120, 1921, 190, 352, 13593],
}

show_table(d)

“Total” 열은 각 결과에 해당하는 임신 건수를 나타냅니다. 이를 확인하기 위해 각 값이 나타나는 횟수를 세는 `value_counts` 메서드와, `Index`의 값(왼쪽 열)에 따라 결과를 정렬하는 `sort_index`를 사용하겠습니다.

In [16]:
preg["outcome"].value_counts().sort_index()

공개된 표와 비교하면 `outcome`의 값이 올바르게 들어왔음을 확인할 수 있습니다. 마찬가지로 다음은 `birthwgt_lb`의 공개된 표입니다.

In [17]:
d = {
    "Value": [".", "0-5", "6", "7", "8", "9-95", "97", "98", "99", "Total"],
    "Label": [
        "inapplicable",
        "UNDER 6 POUNDS",
        "6 POUNDS",
        "7 POUNDS",
        "8 POUNDS",
        "9 POUNDS OR MORE",
        "Not ascertained",
        "REFUSED",
        "DON'T KNOW",
        "",
    ],
    "Total": [4449, 1125, 2223, 3049, 1889, 799, 1, 1, 57, 13593],
}

show_table(d)

출생 체중은 생아 출산으로 끝난 임신에만 기록됩니다. 표를 보면 이 변수가 해당하지 않는 사례가 4,449건입니다. 그 밖에 질문하지 않은 사례가 1건, 응답자가 답하지 않은 사례가 1건, 모른다고 답한 사례가 57건 있습니다.

다시 `value_counts`를 사용해 데이터셋의 빈도를 코드북의 빈도와 비교할 수 있습니다.

In [18]:
counts = preg["birthwgt_lb"].value_counts(dropna=False).sort_index()
counts

`dropna=False`는 `value_counts`가 “NA” 또는 “해당 없음”인 값을 무시하지 않게 합니다. 결과에서 이 값들은 “Not a number”를 뜻하는 `NaN`으로 나타납니다. 그 개수는 코드북의 해당 없음 사례 수와 일치합니다.

6, 7, 8파운드의 빈도도 코드북과 일치합니다. 0~5파운드 범위의 빈도를 확인하려면, “location”을 줄인 이름인 `loc` 속성과 슬라이스 인덱스를 이용해 빈도의 일부를 선택할 수 있습니다.

In [19]:
counts.loc[0:5]

`sum` 메서드로 이 값들을 더할 수 있습니다.

In [20]:
counts.loc[0:5].sum()

합계는 코드북과 일치합니다.

97, 98, 99는 출생 체중을 알 수 없는 사례입니다. 결측 데이터를 다루는 방법은 여러 가지입니다. 간단한 방법은 이 값들을 `NaN`으로 바꾸는 것입니다. 이때 명백히 잘못된 값인 51파운드도 함께 바꾸겠습니다.

`replace` 메서드를 다음과 같이 사용할 수 있습니다.

In [21]:
preg["birthwgt_lb"] = preg["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)

첫 번째 인수는 바꿀 값의 목록입니다. 두 번째 인수인 `np.nan`은 NumPy에서 가져온 `NaN` 값입니다.

이런 자료를 읽을 때는 흔히 오류를 확인하고 특수한 값을 처리해야 합니다. 이런 작업을 **데이터 정제**(data cleaning)라고 합니다.

## 변환

데이터 정제의 또 다른 작업으로, 데이터를 다른 형식으로 바꾸거나 추가 계산을 해야 할 때가 있습니다.

예를 들어 `agepreg`에는 임신이 끝났을 때의 산모 나이가 들어 있습니다. 코드북에 따르면 단위는 100분의 1년이고, 값은 정수입니다. `mean` 메서드로 평균을 구해 보면 이를 알 수 있습니다.

In [22]:
preg["agepreg"].mean()

연 단위로 바꾸려면 모든 값을 100으로 나누면 됩니다.

In [23]:
preg["agepreg"] /= 100.0
preg["agepreg"].mean()

이제 평균이 더 그럴듯해 보입니다.

또 다른 예로, `birthwgt_lb`와 `birthwgt_oz`에는 출생 체중의 파운드 부분과 온스 부분이 서로 다른 열에 들어 있습니다. 이를 하나의 열로 합쳐 파운드 단위의 소수로 나타내면 더 편리합니다.

먼저 `birthwgt_lb`에서 했던 것처럼 `birthwgt_oz`를 정제하겠습니다.

In [24]:
preg["birthwgt_oz"].value_counts(dropna=False).sort_index()

In [25]:
preg["birthwgt_oz"] = preg["birthwgt_oz"].replace([97, 98, 99], np.nan)

이제 정제된 값으로 새 열을 만들 수 있습니다. 파운드와 온스를 합쳐 하나의 양으로 나타냅니다.

In [26]:
preg["totalwgt_lb"] = preg["birthwgt_lb"] + preg["birthwgt_oz"] / 16.0
preg["totalwgt_lb"].mean()

계산한 평균은 그럴듯해 보입니다.

## 요약 통계량

**통계량**(statistic)은 데이터셋에서 구한 수로, 보통 데이터의 어떤 측면을 수치로 나타내기 위한 것입니다. 개수, 평균, 분산, 표준편차가 그 예입니다.

`Series` 객체의 `count` 메서드는 `nan`이 아닌 값의 개수를 반환합니다.

In [27]:
weights = preg["totalwgt_lb"]
n = weights.count()
n

값의 합계를 반환하는 `sum` 메서드도 있습니다. 이를 이용하면 다음처럼 평균을 계산할 수 있습니다.

In [28]:
mean = weights.sum() / n
mean

앞에서 보았듯이, 같은 일을 하는 `mean` 메서드도 있습니다.

In [29]:
weights.mean()

이 데이터셋에서 평균 출생 체중은 약 7.3파운드입니다.

분산은 값들이 퍼져 있는 정도를 수치로 나타내는 통계량입니다. 각 점에서 평균을 뺀 편차를 제곱하고, 그 평균을 구한 것입니다.

In [30]:
squared_deviations = (weights - mean) ** 2

편차 제곱의 평균은 다음과 같이 계산할 수 있습니다.

In [31]:
var = squared_deviations.sum() / n
var

예상할 수 있듯이 `Series`에는 *거의* 같은 일을 하는 `var` 메서드가 있습니다.

In [32]:
weights.var()

결과가 조금 다른 이유는 `var` 메서드가 편차 제곱의 평균을 계산할 때 `n` 대신 `n-1`로 나누기 때문입니다. 무엇을 하려는지에 따라 표본의 분산을 계산하는 두 가지 방법이 있습니다. 차이는 [8장](https://allendowney.github.io/ThinkStats/chap08.html#estimating-variance)에서 설명하겠습니다. 다만 실제로는 대개 큰 차이가 없습니다. 분모가 `n`인 방식을 원한다면 `var` 메서드에 키워드 인수 `ddof=0`을 전달하면 됩니다.

In [33]:
weights.var(ddof=0)

이 데이터셋에서 출생 체중의 분산은 약 1.98입니다. 하지만 이 값은 해석하기 어렵습니다. 우선 단위가 파운드의 제곱입니다. 분산은 일부 계산에 유용하지만, 데이터셋을 설명하는 데 좋은 방법은 아닙니다. 더 나은 선택은 분산의 제곱근인 **표준편차**(standard deviation)입니다. 다음과 같이 계산할 수 있습니다.

In [34]:
std = np.sqrt(var)
std

또는 `std` 메서드를 사용할 수 있습니다.

In [35]:
weights.std(ddof=0)

이 데이터셋에서 출생 체중의 표준편차는 약 1.4파운드입니다. 대략 말하면, 평균에서 표준편차의 한두 배만큼 떨어진 값은 흔하고, 그보다 더 멀리 떨어진 값은 드뭅니다.

## 해석

데이터를 제대로 다루려면 통계와 맥락이라는 두 수준에서 동시에 생각해야 합니다. 예를 들어 임신 파일에서 `caseid`가 10229인 행을 골라 보겠습니다. `query` 메서드는 열 이름, 비교 연산자, 숫자 등을 포함할 수 있는 문자열을 받습니다.

In [36]:
subset = preg.query("caseid == 10229")
subset.shape

결과는 조건이 `True`인 행만 담긴 `DataFrame`입니다. 이 응답자는 임신을 일곱 번 했다고 보고했습니다. 다음은 시간순으로 기록된 임신 결과입니다.

In [37]:
subset["outcome"].values

결과 코드 `1`은 생아 출산을 뜻합니다. `4`는 유산, 즉 대개 의학적 원인을 알 수 없는 임신 소실을 뜻합니다.

통계적으로 이 응답자는 이례적이지 않습니다. 임신 소실은 흔하며, 같은 횟수의 경험을 보고한 다른 응답자도 있습니다. 하지만 맥락을 생각하면 이 데이터에는 한 여성이 여섯 번 임신했고, 매번 유산으로 끝났다는 이야기가 담겨 있습니다. 가장 최근인 일곱 번째 임신은 생아 출산으로 끝났습니다. 공감하는 마음으로 데이터를 본다면 그 이야기에 마음이 움직이는 것은 자연스러운 일입니다.

NSFG 데이터셋의 각 행 뒤에는 사적이고 답하기 어려운 여러 질문에 솔직히 답해 준 사람이 있습니다. 우리는 이 자료로 가족생활, 생식, 건강에 관한 통계적 질문에 답할 수 있습니다. 동시에 데이터가 나타내는 사람들을 생각하고, 존중과 감사를 표할 의무가 있습니다.

## 용어 정리

각 장의 끝에는 그 장에서 정의한 용어를 정리합니다.

- **일화적 근거**(anecdotal evidence): 소수의 개별 사례에서 비공식적으로 수집한 자료. 체계적인 표집 없이 모은 경우가 많습니다.
- **횡단면 연구**(cross-sectional study): 한 시점이나 기간에 모집단의 표본에서 자료를 수집하는 연구.
- **주기**(cycle): 여러 시기에 자료를 수집하는 연구에서 한 차례의 자료 수집 기간.
- **모집단**(population): 연구 대상인 사람이나 항목의 전체 집단.
- **표본**(sample): 모집단의 일부. 무작위로 고르는 경우가 많습니다.
- **응답자**(respondents): 설문조사에 참여해 질문에 답하는 사람들.
- **대표성**(representativeness): 연구 목적에 중요한 면에서 표본이 모집단과 비슷한 성질.
- **과대표집**(oversampling): 보통 타당한 결론을 내릴 만큼 충분한 구성원을 포함하기 위해, 어떤 집단의 구성원이 다른 집단보다 표본에 뽑힐 확률을 높이는 것.
- **변수**(variable): 조사 자료에서 질문에 대한 응답이나 응답으로 계산한 값의 모음.
- **코드북**(codebook): 데이터셋의 변수를 설명하고 자료에 관한 다른 정보도 제공하는 문서.
- **재코딩 변수**(recode): 데이터셋의 다른 변수로 계산한 변수.
- **원자료**(raw data): 수집한 뒤 가공하지 않은 자료.
- **데이터 정제**(data cleaning): 데이터셋의 오류를 찾아 바로잡고, 결측값을 처리하고, 재코딩 변수를 계산하는 과정.
- **통계량**(statistic): 표본의 성질을 설명하거나 요약하는 값.
- **표준편차**(standard deviation): 데이터가 평균 주위에 퍼져 있는 정도를 수치로 나타내는 통계량.

## 연습문제

이 장의 연습문제는 NSFG 임신 파일을 바탕으로 합니다.

### 연습문제 1.1

`preg`에서 `birthord` 열을 골라 값별 빈도를 출력하고, [코드북](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NSFG/Cycle6Codebook-Pregnancy.pdf)에 공개된 결과와 비교하세요.

In [38]:
# Solution goes here

### 연습문제 1.2

출생 체중을 킬로그램으로 나타내는 `totalwgt_kg` 열을 새로 만드세요. 1킬로그램은 약 2.2파운드입니다. 새 열의 평균과 표준편차를 구하세요.

In [39]:
# Solution goes here

### 연습문제 1.3

`caseid`가 2298인 응답자의 임신 기간은 각각 얼마인가요?

In [40]:
# Solution goes here

`caseid`가 5013인 응답자가 낳은 첫아이의 출생 체중은 얼마인가요? 힌트: 쿼리에서 `and`를 사용하면 여러 조건을 함께 확인할 수 있습니다.

In [41]:
# Solution goes here

In [42]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT License](https://mit-license.org/)

본문 라이선스: [Creative Commons 저작자표시-비영리-동일조건변경허락 4.0 국제](https://creativecommons.org/licenses/by-nc-sa/4.0/). 이 한국어 번역에도 같은 본문 라이선스를 적용합니다.